# 03. Feature Engineering — 共通 → 一部共通 → モデル別

4モデル(LightGBM / XGBoost / CatBoost / RealMLP)が本番で使っている特徴量を、
**作る順番どおりに上から実行して確かめる**ノートブック。
各節では「何をするか」に加えて **「なぜするか(意図)」** と、その根拠になった測定値を書く。

## このノートブックの流れ

| 章 | 内容 | 対象モデル |
|---|---|---|
| 1 | **EDA の仮説と、この工程での検証**(何を確かめ、どうなったか) | — |
| 2 | 準備(読み込み・fold 分割) | — |
| 3 | **全モデル共通**の Feature Engineering: 厳密値キー / Smooth Keys / digit features | 4モデル |
| 4 | **一部のモデルで共通**の Feature Engineering: 厳密値 Target Encoding / Count Encoding | GBDT 3種 / LightGBM・XGBoost |
| 5 | **モデル別**の Feature Engineering と、組み上がった列が本番と一致するかの確認 | 各モデル |
| 6 | スコア(04 の実行結果のメモ) | — |
| 7 | `.py` との構成の違い | — |
| 8 | 不採用にした施策(EDA の仮説から検証したものは詳しく) | — |

- 関数を呼んだら**すぐ下で print して中身を確かめる**。
- 目的変数を使う Feature Engineering(Target Encoding)は、本番と同じく **fold 1 の学習行だけで作る**。
- **学習はここでは行わない。** 学習・評価は `src/04_train_and_evaluate_<model>.py` で定義・実行する(6章)。
- 改善幅は OOF AUC の差。採用基準は **+0.00008 以上かつ paired DeLong 検定で z ≥ 3**
  (2026-09-21 以前の施策は旧基準 +0.0002 で判定)。

## 1. EDA の仮説と、この工程での検証

`01_eda.ipynb` の結論で、次工程に3つの施策を渡した(詳細は `docs/eda_results.md`)。

| EDA の施策 | 元になった仮説 | 検証した工程 |
|---|---|---|
| 施策1 カテゴリ列を落とさずモデルに渡す | 仮説1: 補助金・航続距離不安などカテゴリ列も強く効く | `02_baseline`で検証し、効果ありと確認済 |
| **施策2 値を丸めずに購入率を渡す、ビン数を引き上げる** | 仮説2: 年収は細かい値そのものに情報がある | **この工程(3〜5章)** |
| **施策3 交互作用の特徴量は、作る前に本当に必要かを測る** | 仮説4・6: 交互作用は「自宅充電 × スタンド数」の1組 | **この工程(8-1)** |

加えて、EDA の所見から、元の説明変数のうち、効果が薄いとされる「列を減らす」「補助金の有無を他の説明変数と組み合わせる」こともあわせて検証した。

### 仮説 → 施策 → 結果

| 仮説(EDA) | 施策 | 結果(OOF AUC の差) | 判定 | 節 |
|---|---|---|---|---|
| 仮説1 カテゴリ列も効く | カテゴリ6列をモデルに渡す | 0.8838 → 0.9412(**+0.058**) | **採用** | `02_baseline` |
| 仮説2 年収の細かい値に情報がある | 厳密値キーで Target Encoding | **+0.00345**(LightGBM)/ +0.00108(CatBoost) | **採用** | 3-1 / 4-1 |
| 同上(丸めずに見せる別の経路) | ビン数 1024 + digit features + Smooth Keys + 平滑化3種の TE | +0.00052 / +0.00048 / **+0.00096**(LGBM / XGB / CatBoost) | **採用** | 3-2 / 3-3 / 4-1 |
| 仮説4・6 交互作用は1組だけ | 組み合わせの特徴量 | -0.00004。**新たに特徴量を生成しなくても、全モデルが既にこの交互作用を再現していた** | 不採用 | 8-1 |
| 仮説4 `City_Type` はスタンド数と重複 | `City_Type` を削除 | -0.000017(z=-1.30) | 誤差 → 残す | 8-1 |
| 仮説1・3 年齢・性別・保有台数は効かない | この3列を削除 | **-0.00044(z=-14.19)** | **悪化** | 8-1 |
| 仮説5 補助金は誰にでも一律に効く | 補助金との組み合わせ(TE・積) | -0.000016 / +0.000009 | 誤差 → 単体のまま | 8-1 |
| (発展)NN なら組み合わせを取りこぼしているのでは | RealMLP の取りこぼし診断 | 全構造で取りこぼしなし | 追加不要 | 8-1 |

**効いたのは「情報を失わずにモデルへ渡す」方向の施策(施策1・2)。**
「組み合わせる」「減らす」方向はどれも効かなかった。EDA で分かった「各列の効果が log-odds 上で足し算になっている」というデータの構造と整合する。


> 数値カラムの細かい情報（年収）は、グループとしてまとめずに細かく定義したほうが予測精度が向上する。

> 元の情報は削除せず使用する。

> 説明変数同士の組み合わせは、特徴量として新たに生成しなくてもモデルが分類できている。



## 2. 準備

In [1]:
import os, sys, importlib, time
# リポジトリルートを作業ディレクトリにして、data/ などの相対パスを揃える
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.path.abspath("src"))

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
import feature_catalog as fc

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)


def load_fe(name):
    # src/03_feature_engineering_<name>.py を読み込む(数字で始まるファイル名は import 文で書けないため)
    return importlib.import_module(f"03_feature_engineering_{name}")


fe_lgbm, fe_xgb, fe_cb, fe_mlp = (load_fe(n) for n in ("lgbm", "xgb", "catboost", "realmlp"))
# 絶対パスは出力しない(公開リポジトリに個人のディレクトリ構成を残さないため)
print("data/ を検出:", os.path.isdir("data"))

data/ を検出: True


In [2]:
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
y = (train["Will_Buy_EV"] == "Yes").astype(int).to_numpy()

NUM, CAT = fe_lgbm.NUMERIC_COLS, fe_lgbm.CATEGORICAL_COLS

# fold 分割は全モデル共通(変更禁止)。Feature Engineering の実演には fold 1 を使う
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
tr_idx, va_idx = next(iter(skf.split(train, y)))

print(f"train {train.shape} / test {test.shape} / 購入率 {y.mean():.3f}")
print(f"fold 1: 学習 {len(tr_idx):,} 行 / 検証 {len(va_idx):,} 行")
print("数値列    :", NUM)
print("カテゴリ列:", CAT)

train (668665, 15) / test (286571, 14) / 購入率 0.175
fold 1: 学習 534,932 行 / 検証 133,733 行
数値列    : ['Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned', 'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work', 'Environmental_Concern_Level']
カテゴリ列: ['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level']


各章の最後で、組み上げた列が本番と同じかを確かめる。
本番の列名は `04_train_and_evaluate_<model>.py --dump-features` が書き出した `docs/features_<model>.json` にある。

In [3]:
def check_columns(tag, columns):
    # 作った列が本番(docs/features_<tag>.json)と一致するかを確かめる
    expected = fc.load(tag)["columns"]
    cols = list(columns)
    same_set = set(cols) == set(expected)
    same_order = cols == expected
    print(f"{tag}: {len(cols)} 列(本番 {len(expected)} 列)"
          f" → 列の集合 {'一致' if same_set else '不一致'} / 並び順 {'一致' if same_order else '不一致'}")
    if not same_set:
        print("  ノートブックだけにある列:", sorted(set(cols) - set(expected)))
        print("  本番だけにある列      :", sorted(set(expected) - set(cols)))

## 3. 全モデル共通の Feature Engineering

どのモデルにも入っている3つ。**考え方は共通だが、実装の細部はモデルごとに違う**(各節の表)。
3つはどれも「年収の細かい値を、モデルに失わせずに届ける」ための部品で、役割分担がある(3-4 で整理する)。

### 3-1 厳密値キー — 年収の「値そのもの」をキーにする(仮説2)

| 厳密値キー を加えないときのデメリット | 理由 |
|---|---|
| **細かい値が同じ扱いになる** | 木系のモデルは学習前に値を 255 個のビンにまとめ、ビンの境目でしか分割しない。年収 13,214 種類では平均 52 種類の値が1つのビンに入る |
| **値ごとの購入率を1回の分割で表せない** | 木の分割は「年収 ≧ ○○ か」という大小の境目だけ。値ごとに購入率が上下していると、1つの値を切り出すのに境目が2本要る。深さ 5 の木では 13,214 種類を切り分けられない |

そこで、値そのものをキーにして「**その値の人の購入率**」を1列にする(Target Encoding、4-1)。
購入率の列なら、**「購入率 ≧ 20% か」の1回の分割で、購入率の高い値と低い値をまとめて分けられる。**
キーを作る時点で丸めると値ごとの違いが消えるので、数値列も値のままキーにする(EDA の仮説2)。
小数1桁の値の揺れ(23.4 が 23.39999… になるなど)を防ぐため、×10 して整数にしてから使う。

| モデル | 使い方 |
|---|---|
| GBDT 3種 | このキーで Target Encoding / Count Encoding を作る(4章) |
| RealMLP | 値をそのままカテゴリ(`{列名}_cat_`)として embedding に渡す(5-4) |

In [4]:
keys_tr, keys_te = fe_lgbm.make_key_frame(train, test)

rows_per_value = {c: keys_tr[c].value_counts() for c in keys_tr.columns}
summary = pd.DataFrame({
    "種類数": keys_tr.nunique(),
    "1値あたりの行数(平均)": {c: round(v.mean(), 1) for c, v in rows_per_value.items()},
    "1値あたりの行数(中央値)": {c: v.median() for c, v in rows_per_value.items()},
})
print(summary.to_string())

inc_counts = rows_per_value["Annual_Income_USD"]
print()
print(f"年収: 10 行未満の値が {(inc_counts < 10).mean():.1%} / "
      f"最小値 {train['Annual_Income_USD'].min():,.0f} ドルだけで全体の {(train['Annual_Income_USD'] == train['Annual_Income_USD'].min()).mean():.1%}")

                               種類数  1値あたりの行数(平均)  1値あたりの行数(中央値)
Age                             45       14859.2        14600.0
Annual_Income_USD            13214          50.6            8.0
Daily_Commute_km               805         830.6          576.0
Number_of_Cars_Owned             4      167166.2       178444.5
Charging_Stations_Near_Home     15       44577.7        52986.0
Charging_Stations_Near_Work     20       33433.2        22861.0
Environmental_Concern_Level      5      133733.0       130469.0
Gender                           3      222888.3       295427.0
City_Type                        3      222888.3       255377.0
Current_Car_Type                 4      167166.2       162991.5
Home_Charging_Possible           2      334332.5       334332.5
Subsidy_Available                2      334332.5       334332.5
Range_Anxiety_Level              3      222888.3        62499.0

年収: 10 行未満の値が 50.8% / 最小値 30,000 ドルだけで全体の 9.2%


**年収は「1値あたり平均 50 行」だが、中央値は 8 行しかない。** 平均を押し上げているのは、
最小値 30,000 ドルに全体の 9% が集まっている部分。
**半分の値は 10 行未満**で、その値だけの購入率はノイズが大きい。これを補うのが次の Smooth Keys。

### 3-2 Smooth Keys — 同じ年収を、粗い解像度でも見せる

**意図**: 厳密値キーだけだと、行数の少ない値(年収の半分)は購入率の推定が不安定になる。
そこで「年収 /100(100 ドル刻み)」「/1000」のように**近い値をまとめたキーも併せて**持たせ、行数の少ない値でも近い値の購入率を参照できるようにする。
**厳密値キーを置き換えるのではなく足す。** 

改善幅は単独では測っておらず、平滑化3種の TE・digit・ビン数 1024 とまとめて Run 9 で採用した(+0.00048〜0.00096)。

| モデル | 年収 | 通勤距離 | 使い方 |
|---|---|---|---|
| LightGBM | /10・/100・/1000 | floor(km) | TE / Count のキー |
| XGBoost | /100・/1000・/10000 | floor(km) | TE のキー |
| CatBoost | /1・/100・/1000 | floor(km) | TE のキー(/1 は厳密値と同じで重複だが実害なし) |
| RealMLP | /100・/1000・/10000 | floor(km / 5) | カテゴリとして embedding に渡す |

下は LightGBM 版。

In [5]:
keys_tr = fe_lgbm.add_smooth_keys(keys_tr, train)
keys_te = fe_lgbm.add_smooth_keys(keys_te, test)

sample = pd.concat([train[["Annual_Income_USD", "Daily_Commute_km"]],
                    keys_tr[fe_lgbm.SMOOTH_KEYS]], axis=1)
print(sample.head().to_string())
print()
print("種類数:", keys_tr[fe_lgbm.SMOOTH_KEYS].nunique().to_dict())
print("1値あたりの行数(中央値):",
      {c: int(keys_tr[c].value_counts().median()) for c in fe_lgbm.SMOOTH_KEYS})

   Annual_Income_USD  Daily_Commute_km  sk_inc10  sk_inc100  sk_inc1000  sk_commute
0            92887.0              23.4      9288        928          92          23
1            30000.0               5.0      3000        300          30           5
2            94389.0              36.8      9438        943          94          36
3            73580.0              23.7      7358        735          73          23
4            57898.0              50.8      5789        578          57          50

種類数: {'sk_inc10': 5933, 'sk_inc100': 1142, 'sk_inc1000': 150, 'sk_commute': 89}
1値あたりの行数(中央値): {'sk_inc10': 55, 'sk_inc100': 298, 'sk_inc1000': 3088, 'sk_commute': 5429}


年収の1値あたりの行数(中央値)は、厳密値の 8 行から /10 で 55 行、/100 で約 300 行、/1000 で約 3,000 行に増える。
粗くするほど推定は安定するが、値ごとの違いは見えにくくなる。その両方を並べて渡している。

### 3-3 digit features — 数値を桁ごとの列にばらす

**意図**: 木モデルは学習前に数値列を**ビン**(LightGBM・XGBoost は既定 255 個、CatBoost は本番の軽量設定で 64 個)にまとめ、
ビンの境目でしか分割しない。年収 13,214 種類は 255 個に押し込まれ、**平均 52 種類の値が同じ扱い**になる。
年収 92,887 を「1の位 7 / 10の位 8 / 100の位 8 / 1000の位 2 …」と桁ごとの列にすると、
**各列は 0〜9 の10種類しかないのでビンで潰れない。** 木は桁の列を順に分割していけば、元の値まで辿り着ける。

| モデル | 列数 | 違い |
|---|---|---|
| LightGBM | 15 | 浮動小数で計算し、train で定数の桁を落とす |
| XGBoost / CatBoost | 16 | 整数に直してから計算し、train と test の両方で定数の桁だけ落とす |
| RealMLP | 1 | 通勤距離の小数部(`_Daily_Commute_km_decimal`)のみ。値そのものを embedding で持つので桁は不要 |

**正直な測定結果(Run 14、平滑化3種の TE なしで digit だけを足した場合)**

| モデル | digit のみ | ビン数を上げるだけ | 両方 |
|---|---|---|---|
| LightGBM(ビン 255 → 1024) | +0.00010 | +0.00001 | +0.00011 |
| XGBoost(ビン 255 → 1024) | +0.00009 | ±0 | +0.00007 |
| CatBoost(ビン 64 → 1024) | **+0.00044** | **+0.00038** | **+0.00061** |

**LightGBM・XGBoost では単独の効果は誤差だった。** 厳密値 TE が値ごとの購入率を既に渡しているため。
**CatBoost でははっきり効いた。** ビンが 64 個と最も粗く、年収の値の違いが最も潰れていたため。
LightGBM・XGBoost でも入れたままにしているのは、悪化はせず、Run 9 の一式(+0.0005)の一部として採用したから。

下は LightGBM 版。

In [6]:
digit_tr = fe_lgbm.add_digit_features(train)
digit_te = fe_lgbm.add_digit_features(test, keep=list(digit_tr.columns))  # test も同じ列にそろえる

inc_digits = [c for c in digit_tr.columns if c.startswith("Annual_Income_USD")]
print(pd.concat([train["Annual_Income_USD"], digit_tr[inc_digits]], axis=1).head().to_string())
print()
print(f"digit 列: {digit_tr.shape[1]} 列 →", list(digit_tr.columns))

   Annual_Income_USD  Annual_Income_USD_digit0  Annual_Income_USD_digit1  Annual_Income_USD_digit2  Annual_Income_USD_digit3
0            92887.0                         7                         8                         8                         2
1            30000.0                         0                         0                         0                         0
2            94389.0                         9                         8                         3                         4
3            73580.0                         0                         8                         5                         3
4            57898.0                         8                         9                         8                         7

digit 列: 15 列 → ['Age_digit0', 'Age_digit1', 'Annual_Income_USD_digit0', 'Annual_Income_USD_digit1', 'Annual_Income_USD_digit2', 'Annual_Income_USD_digit3', 'Daily_Commute_km_digit-1', 'Daily_Commute_km_digit0', 'Daily_Commute_km_digit1', 'Number_of

### 3-4 厳密値・Smooth Keys・digit は矛盾しないのか

「値を丸めずに使う」と言いながら、粗く丸めたキーや桁への分解も足しているのは矛盾に見える。
**3つは同じ目的(値の細部を失わせない)に対する、弱点の違う経路**として併用している。

| 部品 | 何を渡すか | 強み | 弱み(次の部品が補う) |
|---|---|---|---|
| 厳密値キー → TE | 「その値ちょうど」の購入率 | 値ごとの違いを1列で直接渡せる | 行数の少ない値(年収の半分は 10 行未満)では推定がぶれる |
| Smooth Keys → TE | 「近い値の範囲」の購入率 | 行数が増えて推定が安定する | 値ごとの細かい違いは見えない |
| digit features | 値そのもの(ラベルを使わない形) | ビンで潰れない。**目的変数を通さない**ので TE のぶれと無関係 | 1列だけでは意味を持たず、効果も小さい(CatBoost 以外は誤差) |

- **Smooth Keys は「丸めて置き換える」ではなく「粗い見方を足す」。** 厳密値キーはそのまま残っている
- **digit features は「ビンに丸める」の逆。** ビンで潰れてしまう値の細部を、桁に分けることで木に見せる
- 3つとも並べて渡し、**値ごとにどれを信じるかは木が選ぶ**。平滑化の強さを3種類並べる(4-1)のも同じ考え方

## 4. 一部のモデルで共通の Feature Engineering

### 4-1 厳密値 Target Encoding(GBDT 3種)— 最大の改善要因(仮説2)

**意図**: 木は値をビンにまとめてから分割するので、「年収 92,887 ドルの人の購入率」を直接は学べない。
**値ごとの購入率を1列にして渡せば、木はその列を1回分割するだけで値ごとの違いを使える。**
改善幅は **+0.00345**(LightGBM、3-fold)。前コンペ(S6E8)でも同じ施策が唯一桁違いに効いた。

**目的変数を使うので、リークを防ぐ作法が要る。** それぞれの理由も書いておく。

| 作法 | やり方 | なぜ必要か |
|---|---|---|
| fold 内で作る | fold 1 の学習行だけで購入率を集計し、検証行・test にはその値を当てる | 検証行の正解が混ざると、検証スコアが実際より良く見える |
| **入れ子 CV** | 学習行自身の値は、さらに内側で 5 分割して**自分を含まない行から**計算する | 自分の正解を含む購入率は、学習時だけ当たりすぎる。木がそれを過信する。入れ子にすると **+0.00108**(XGBoost) |
| 平滑化 | 行数の少ない値は、全体の購入率(17.5%)に寄せる | 3 行しかない値の購入率 0% や 100% を、そのまま信じないため |
| **平滑化を3種類同時に入れる** | 強さ auto / 10 / 100 の3列を並べる(選ぶのではない) | 行数の多い値では弱い平滑化、少ない値では強い平滑化を、**木が値ごとに使い分けられる** |

キーは生の13列 + Smooth Keys 4本 = 17本。17 × 3 = **51 列**になる。

> **注意: 木を弱くしてからでないと逆効果になる。** 深さ無制限・全列を使う設定では、
> 同じ TE が **-0.00831** と大きく悪化した。年収の TE はノイズを含むので、強い木はそれを目印に葉を切り刻む。
> 本番の「深さ 5 + 1本の木が見る列を 30% に絞る」で、1本の木が TE に依存しきれない状態にして初めて効く。

下は LightGBM 版(3モデルとも考え方は同じ)。

In [7]:
te_keys = fe_lgbm.single_keys("all") + fe_lgbm.SMOOTH_KEYS
t0 = time.time()
te_tr, (te_va, te_te) = fe_lgbm.target_encode_fold(
    keys_tr.iloc[tr_idx], y[tr_idx],
    [keys_tr.iloc[va_idx], keys_te],
    te_keys, smooth=["auto", 10.0, 100.0], n_inner=5, seed=42,
)
print(f"{len(te_keys)} キー × 3 種類の平滑化 = {te_tr.shape[1]} 列({time.time() - t0:.0f} 秒)")
print()
print(te_va.iloc[:3, :6].round(4).to_string())

17 キー × 3 種類の平滑化 = 51 列(2 秒)

   te_Age_sauto  te_Age_s10  te_Age_s100  te_Annual_Income_USD_sauto  te_Annual_Income_USD_s10  te_Annual_Income_USD_s100
1        0.1514      0.1515       0.1517                      0.0445                    0.0445                     0.0447
2        0.2068      0.2068       0.2066                      0.4005                    0.3622                     0.2408
6        0.2064      0.2063       0.2061                      0.1746                    0.1746                     0.1746


**リークがないことの確認。** 学習行に自分の正解が混ざっていれば、学習行での AUC だけが不自然に高くなる。
学習行と検証行で、TE 列1本だけの AUC がほぼ同じなら問題ない。

In [8]:
auto_cols = [c for c in te_va.columns if "auto" in c]
check = pd.DataFrame({
    "学習行での AUC": [roc_auc_score(y[tr_idx], te_tr[c]) for c in auto_cols],
    "検証行での AUC": [roc_auc_score(y[va_idx], te_va[c]) for c in auto_cols],
}, index=auto_cols).sort_values("検証行での AUC", ascending=False)
print(check.round(4).head(8).to_string())

                                      学習行での AUC  検証行での AUC
te_Environmental_Concern_Level_sauto     0.8435     0.8409
te_Subsidy_Available_sauto               0.7177     0.7174
te_sk_inc10_sauto                        0.7096     0.7137
te_Annual_Income_USD_sauto               0.7070     0.7120
te_sk_inc100_sauto                       0.7031     0.7039
te_sk_inc1000_sauto                      0.6841     0.6830
te_sk_commute_sauto                      0.5490     0.5533
te_Daily_Commute_km_sauto                0.5488     0.5530


学習行と検証行の AUC の差は最大でも 0.005 程度で、学習行だけが高いという偏りもない。入れ子 CV が効いている。

### 4-2 Count Encoding(LightGBM・XGBoost)— +0.00083 / +0.00049

**意図**: 各キーの値が**何回出てくるか**(頻度)を列にする。役割は2つある。

- **TE の信頼度を木に教える。** 同じ購入率 20% でも、1,000 行から出た 20% と 3 行から出た 20% では重みが違う。
  頻度の列があれば、木は「行数が多い値の TE だけ信じる」という分割ができる
- **頻度そのものに生成過程の痕跡がある。** 年収 30,000 ドルは全体の 9% を占める(3-1)。
  頻度の列はこうした特別な値を1回の分割で切り出せる

目的変数を使わないので、train と test をまとめて数えてよい(リークしない)。
TE と併用すると効果が足し算になった(TE のみ +0.00345 → 併用 +0.00395)。**TE とは別の情報を運んでいる**裏付け。
CatBoost では **-0.00017** と効かなかった。CatBoost は内部で同種の統計を自動で作るため。

In [9]:
cnt_tr, cnt_te = fe_lgbm.count_encode(keys_tr, keys_te, fe_lgbm.single_keys("all"))

cnt_show = [c for c in cnt_tr.columns if "Annual_Income_USD" in c or "City_Type" in c]
print(pd.concat([train[["Annual_Income_USD", "City_Type"]], cnt_tr[cnt_show]], axis=1).head().to_string())
print()
print(f"Count 列: {cnt_tr.shape[1]} 列")

   Annual_Income_USD City_Type  cnt_Annual_Income_USD  cnt_City_Type
0            92887.0  Suburban               0.000156       0.382160
1            30000.0     Rural               0.091999       0.185583
2            94389.0     Urban               0.000063       0.432257
3            73580.0  Suburban               0.000263       0.382160
4            57898.0  Suburban               0.000433       0.382160

Count 列: 13 列


## 5. モデル別の Feature Engineering

ここまでの部品に、モデル固有の処理を足して fold 1 の学習行列を組み上げ、本番の列と突き合わせる。

### 5-1 LightGBM — カテゴリ列を native category で渡す

**意図**: カテゴリ列(2〜4種類)を `category` 型のまま渡すと、LightGBM は値をグループに分けて分割できる。
整数コード(ordinal)でも測ったが **-0.000005(z=-0.42)で差がなかった**(Run 21)。
カテゴリ列の情報は TE 経由でも渡っているため、渡し方の違いが効かない。最も素直な native category のままにした。

組み上げ = 生の13列 + digit(3-3)+ Count(4-2)+ TE(4-1)。

In [10]:
train_c, test_c = fe_lgbm.make_categorical(train, test)

X_static = pd.concat([train_c[NUM + CAT], digit_tr, cnt_tr], axis=1)
X_tr_lgbm = pd.concat([X_static.iloc[tr_idx].reset_index(drop=True),
                       te_tr.reset_index(drop=True)], axis=1)

print("category 型で渡す列:", [c for c in X_tr_lgbm.columns if str(X_tr_lgbm[c].dtype) == "category"])
check_columns("lgbm", X_tr_lgbm.columns)

category 型で渡す列: ['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level']
lgbm: 92 列(本番 92 列) → 列の集合 一致 / 並び順 一致


### 5-2 XGBoost — カテゴリ列を整数コード(ordinal)で渡す

**意図: 精度ではなく多様性のため。** native category / ordinal / One-Hot の3方式は単体スコアが同点だった
(ordinal +0.00009、One-Hot +0.00005 で、どれも誤差)。
アンサンブルでは**同じ予測をするモデルを2つ入れても伸びない**ので、LightGBM(native category)と
違う渡し方にして、予測の癖をずらした。
部品は同じだが、関数は XGBoost 用のファイルのものを使う(Smooth Keys の刻みと digit の計算方法が違う)。

In [11]:
ALL = fe_xgb.NUMERIC_COLS + fe_xgb.CATEGORICAL_COLS
xtr, xte = train[ALL].copy(), test[ALL].copy()
xtr, xte = fe_xgb.add_digit_features(xtr, xte, train, test)
xtr, xte = fe_xgb.add_count_encoding(xtr, xte, train, test, ALL)
xtr, xte = fe_xgb.as_ordinal(xtr, xte, fe_xgb.CATEGORICAL_COLS)
print("カテゴリ列の中身(整数コード):", {c: sorted(xtr[c].unique().tolist()) for c in ["City_Type", "Range_Anxiety_Level"]})

# Target Encoding: Smooth Keys 4本 + 生の13列 = 17 キー × 3 種類の平滑化
sk_tr, sk_te = fe_xgb.make_smooth_keys(train, test)
xte_cols = list(sk_tr.columns) + ALL
codes = fe_xgb.prepare_te_codes(pd.concat([sk_tr, train[ALL]], axis=1),
                                pd.concat([sk_te, test[ALL]], axis=1), xte_cols)
c_tr, c_te, ncats = codes
t0 = time.time()
xte_tr, xte_va, xte_te = fe_xgb.fit_apply_te_cv_nested_multi(
    c_tr, c_te, ncats, pd.Series(y), xte_cols, tr_idx, va_idx, ("auto", 10.0, 100.0), 1)
print(f"TE {xte_tr.shape[1]} 列({time.time() - t0:.0f} 秒)")

X_tr_xgb = pd.concat([xtr.iloc[tr_idx].reset_index(drop=True), xte_tr.reset_index(drop=True)], axis=1)
check_columns("xgb", X_tr_xgb.columns)

カテゴリ列の中身(整数コード): {'City_Type': [0, 1, 2], 'Range_Anxiety_Level': [0, 1, 2]}


TE 51 列(2 秒)
xgb: 93 列(本番 93 列) → 列の集合 一致 / 並び順 一致


### 5-3 CatBoost — catify(低カーディナリティの数値列もカテゴリとして渡す)— +0.00170

**意図**: CatBoost は `cat_features` に指定した列に対して、内部で **Ordered Target Statistics**
(自分より前の行だけで計算する、リークしない購入率)を自動で作る。
年齢(45種)・スタンド数(15〜20種)など**種類の少ない数値列を文字列にして渡せば、その仕組みを数値列にも使える。**
CatBoost 単体で最大の改善(+0.00170)。

**LightGBM に横展開すると -0.00016 と逆効果だった。** 効果の源はカテゴリ化そのものではなく
CatBoost の Ordered Target Statistics にあり、LightGBM ではむしろ値の大小の順序を捨てる分だけ損をした。
このため CatBoost 専用。Count Encoding も内部の統計と重なるので使わない(4-2)。

In [12]:
base = fe_cb.NUMERIC_COLS + fe_cb.CATEGORICAL_COLS
cX, cX_te = train[base].copy(), test[base].copy()
cb_features = list(base)

digit_cols = fe_cb.add_digits(cX)                  # catify で文字列にする前に作る
fe_cb.add_digits(cX_te)
cb_features += fe_cb.drop_constant([cX, cX_te], digit_cols)

sk_cols = fe_cb.add_smooth_keys(cX)                # TE のキー専用(モデルには渡さない)
fe_cb.add_smooth_keys(cX_te)

fe_cb.cast_to_str([cX, cX_te], fe_cb.LOWCARD_NUM_COLS)   # catify
cb_cats = fe_cb.CATEGORICAL_COLS + fe_cb.LOWCARD_NUM_COLS
print("cat_features:", cb_cats)
print("catify 後の Age の型:", cX["Age"].dtype, "/ 例:", cX["Age"].head(3).tolist())

cb_te_cols = list(dict.fromkeys(base + sk_cols))
keep = list(dict.fromkeys(cb_features + sk_cols + cb_te_cols))
cb_tr, cb_va, cb_te = cX[keep].iloc[tr_idx].copy(), cX[keep].iloc[va_idx].copy(), cX_te[keep].copy()
t0 = time.time()
new_te = fe_cb.target_encode(cb_tr, cb_va, cb_te, y[tr_idx], cb_te_cols,
                             smooth=20.0, smooths=[10.0, 20.0, 100.0])   # CatBoost の平滑化3種は 10/20/100
print(f"TE {len(new_te)} 列({time.time() - t0:.0f} 秒)")

check_columns("catboost", cb_features + new_te)

cat_features: ['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level', 'Age', 'Number_of_Cars_Owned', 'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work', 'Environmental_Concern_Level']
catify 後の Age の型: str / 例: ['66', '38', '26']


TE 51 列(11 秒)
catboost: 80 列(本番 80 列) → 列の集合 一致 / 並び順 一致


### 5-4 RealMLP — GBDT とは別系統の前処理(多様性の供給源)

**意図: アンサンブルの中で、GBDT と違う間違え方をするモデルを置くこと。**
GBDT 3種は同じ厳密値 TE を中心にしているため予測が似通い、互いに足しても伸びにくい。
RealMLP は公開カーネル(yekenot)の前処理を移植した、**GBDT とは別の情報の渡し方**をする。

実際、RealMLP に GBDT と同じ厳密値 TE を足すと**単体は +0.000156 伸びたが、アンサンブルへの寄与はゼロ**になった
(GBDT と予測が似て多様性が減る)。単体スコアより「違う見方」を優先して、あえて別系統のままにしている。

| 種類 | 列 | 意図 |
|---|---|---|
| 値のカテゴリ化 | 数値7列を `{列名}_cat_` として embedding へ | 厳密値キー(3-1)の NN 版。値ごとに別のベクトルを学ぶ |
| Smooth Keys | 年収 /100・/1000・/10000、通勤距離 /5 | 3-2 と同じ。行数の少ない値を粗い解像度で補う |
| 分位ビン | 年収を 400 / 600 / 800 / 900 / 1100 分位で区切る | 行数がそろった区切り。解像度の違うビンを並べる |
| フラグ・比 | 年収が10の倍数か / 通勤距離 ÷ 年齢 / 通勤距離の小数部 | 公開カーネルの構成をそのまま移植 |
| 元データ由来 | 元データ(1万行)での年収ごとの購入率(`org_mean`) | コンペデータと別の情報源。アンサンブル +0.000022 |
| 組み合わせ TE | 年収 × 航続距離の不安、年齢 × 航続距離の不安 | 公開カーネルの構成をそのまま移植(fold 内で sklearn の TargetEncoder) |

In [13]:
import warnings
from sklearn.preprocessing import TargetEncoder

# 本番(04)と同じ呼び方をすると、動作に影響しない警告が出るので表示だけ止める。
#   - pandas 3 の select_dtypes の仕様変更予告 / sklearn TargetEncoder の引数の非推奨予告
#   - 分位ビン: 同じ年収の人が多く、幅ゼロの区切りがまとめられたという通知
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*select_dtypes.*")
warnings.filterwarnings("ignore", message="Bins whose width are too small.*")

orig = fe_mlp.load_orig("data/EV_Adoption_and_Range_Anxiety_Dataset.csv")
mX = train.drop(columns=["id", "Will_Buy_EV"])
mX_te = test.drop(columns=["id"])
m_cat = mX.select_dtypes(include=["object"]).columns.tolist()
m_num = mX.select_dtypes(exclude=["object"]).columns.tolist()

category_map = {}
mX, new_cat, new_num, combos = fe_mlp.build_features(mX, m_cat, m_num, category_map, fit=True, orig=orig)
mX_te, _, _, _ = fe_mlp.build_features(mX_te, m_cat, m_num, category_map, fit=False, orig=orig)
print(f"カテゴリとして渡す列 {len(m_cat + new_cat)} 本 / 数値として渡す列 {len(m_num + new_num)} 本")
print("新しく作った数値列:", new_num)

# 組み合わせ TE(fold 内。学習行は内側 5 分割の OOF 値)
m_tr, m_va = mX.iloc[tr_idx].copy(), mX.iloc[va_idx].copy()
enc = TargetEncoder(cv=5, smooth="auto", shuffle=True, random_state=42)
te_names = [f"_{c}TE" for c in combos]
m_tr[te_names] = enc.fit_transform(m_tr[combos], y[tr_idx])
m_va[te_names] = enc.transform(m_va[combos])
print(m_va[te_names].describe().loc[["mean", "min", "max"]].round(4).to_string())

check_columns("realmlp", m_tr.columns)

カテゴリとして渡す列 25 本 / 数値として渡す列 11 本
新しく作った数値列: ['_Daily_Commute_km_/_Age', '_Daily_Commute_km_decimal', '_Annual_Income_USD_mean_target_orig', '_Annual_Income_USD_count']


      _Annual_Income_USD_Range_Anxiety_Level_TE  _Age_Range_Anxiety_Level_TE
mean                                     0.1737                       0.1745
min                                      0.0000                       0.0000
max                                      1.0000                       0.2352
realmlp: 38 列(本番 38 列) → 列の集合 一致 / 並び順 一致


### 5-5 採否表と本番の列の突き合わせ

`src/feature_catalog.py` の `FUNC_STATUS`(〇 採用 / ✖ 不採用)が、`docs/features_*.json` の列と矛盾していないかを確かめる。

In [14]:
ng = fc.verify_status()
print("採否表と本番の列の矛盾:", ng if ng else "なし")

採否表と本番の列の矛盾: なし


## 6. スコア(04 の実行結果のメモ)

**学習と評価は `src/04_train_and_evaluate_<model>.py` で定義・実行する。** このノートブックは学習しない。
04 は上と同じ Feature Engineering 関数を import し、5-fold すべてで同じ手順を回したうえで学習・評価する。
下の値は、04 が保存した OOF 予測(`oof/oof_<model>.npy`)から計算したもの(2026-09-26 時点)。

| モデル | OOF AUC | アンサンブル | 1回の所要時間(フル 5-fold) |
|---|---|---|---|
| LightGBM | 0.946095 | 1/3 | 約 7 分 |
| XGBoost | 0.946077 | 1/3 | 約 11 分 |
| RealMLP | 0.945888 | 1/3 | 約 50 分 |
| CatBoost | 0.945890 | 0(他の GBDT と同質で選ばれない) | 約 67 分 |
| **アンサンブル**(順位の平均) | **0.946234** | — | Public LB **0.94645** |

本番の実行コマンド(リポジトリのルートで実行):

```bash
uv run src/04_train_and_evaluate_lgbm.py --patterns base,te1,cnt1,digit,sk --smooths auto,10,100 \
  --max_bin 1024 --feature_fraction 0.3 --max_depth 5 \
  --folds 5 --learning_rate 0.03 --n_estimators 8000 --early_stopping 200 --n_jobs 7 --save --tag lgbm
uv run src/04_train_and_evaluate_xgb.py --pattern tte_sk_dig --max-bin 1024 \
  --set-param colsample_bytree=0.3 --set-param max_depth=5 \
  --folds 5 --learning-rate 0.03 --n-estimators 8000 --early-stopping 200 --n-jobs 7 --save --out-suffix ""
uv run src/04_train_and_evaluate_catboost.py --fe te_all,catify,digits,skeys,te3 \
  --folds 5 --iters 1000 --lr 0.06 --fast --border 64 --hc-border 1024 --threads 7 --save
uv run src/04_train_and_evaluate_realmlp.py --folds 5 --threads 7 --tag realmlp
```

## 7. `.py` との構成の違い

| 観点 | `.py`(`src/03_*.py` / `src/04_*.py`) | このノートブック |
|---|---|---|
| 分け方 | **モデルごと**にファイルを分ける(`03_feature_engineering_lgbm.py` など) | **Feature Engineering の種類ごと**に分ける(全モデル共通 → 一部共通 → モデル別) |
| `03` の役割 | 関数の定義だけを置く。実行コードは書かない | 関数を呼び、すぐ下で print して中身を確かめる。意図と根拠の数字も書く |
| 置いてある関数 | 採用した関数と**不採用の関数が混在**(再検証しないための記録)。採否は `feature_catalog.py` の `FUNC_STATUS` が持つ | **採用した関数だけ**を呼ぶ。不採用の施策は8章に一覧だけ載せる |
| 学習・評価 | `04` が `03` を import し、Feature Engineering → 5-fold 学習 → 評価 → 成果物の保存まで行う | 学習しない。スコアは 04 の結果をメモするだけ(6章) |
| 使うデータ | 5-fold すべて | fold 1 のみ(列の中身と並びを確かめるには十分) |
| 本番との一致の確認 | `--dump-features` で列名を `docs/features_*.json` に書き出す | その json と、ここで作った列を突き合わせる(5章) |
| 所要時間 | 1モデル 7〜67 分 | 1 分程度 |
| 同じ部品の実装 | モデルごとに少しずつ違う(Smooth Keys の刻み、digit の計算方法など) | 違いを各節の表にまとめ、実演は代表(主に LightGBM 版)で行う |

## 8. 不採用にした施策

検証したうえで本番に入れなかった施策。**コードは再検証を防ぐ記録として `src/03_*.py` に残してある**(`FUNC_STATUS` で ✖)。
「変化」は OOF AUC の差(+ が改善)。

### 8-1 EDA の仮説から検証したもの

#### (1) 交互作用: 自宅充電 × 自宅近くのスタンド数(仮説4・6、EDA の施策3)

**仮説**: EDA で唯一見つかった本物の交互作用。自宅で充電できない人だけ、スタンドが多いほど買う
(スタンド 0 → 7 個で log-odds が **+0.78**。自宅で充電できる人は **-0.02**)。
この組み合わせを特徴量として明示すれば効くのではないか。

**検証**: この組を含む3列の組み合わせ TE(`triple_keys`)で **-0.00004**(効果なし)。
さらに、**モデルがこの交互作用を既に学習しているか**を、保存済みの OOF 予測で直接確かめた。

| 群 | 行数 | 観測 | LightGBM | XGBoost | CatBoost | RealMLP |
|---|---|---|---|---|---|---|
| 自宅充電 できない | 205,988 | **+0.778** | +0.737 | +0.738 | +0.726 | +0.689 |
| 自宅充電 できる | 462,677 | **-0.021** | -0.015 | -0.016 | -0.012 | -0.018 |

**4モデルとも、向きも大きさもほぼ正しく再現していた。**
深さ 5 の木は「自宅充電で分けてからスタンド数を見る」という2段の分割で、この交互作用を自力で表現できる。
特徴量として明示しても新しい情報にならない。

#### (2) 列の削除: `City_Type`(仮説4)

**仮説**: `City_Type` はスタンド数からほぼ復元できる(自宅スタンド 8 個以上の 150,407 行のうち、都市居住でないのは 15 行)。
情報が重複しているので、落としても損をしないのではないか。

| 落とした列 | 列数 | 変化 | z | 判定 |
|---|---|---|---|---|
| `City_Type` | 92 → 87 | -0.000017 | -1.30 | 誤差 |

**情報が重複していたのは本当だったが、落としても得はしない。** 構成を変える理由がないので残した。

#### (3) 列の削除: `Age` / `Gender` / `Number_of_Cars_Owned`(仮説1・3)

**仮説**: 3列とも、目的変数との関係が弱く(購入率の幅 4.1 / 0.5 / 1.2 pt)、他の列ともほぼ無関係だった。
ノイズ列なので、落とせば1本の木が有用な列を引く確率が上がるのではないか。

| 落とした列 | 列数 | 変化 | z | 判定 |
|---|---|---|---|---|
| 3列 | 92 → 74 | **-0.000441** | **-14.19** | **明確に悪化** |
| 3列(1本の木が見る列の数をそろえて再測定) | 92 → 74 | -0.000432 | -13.99 | 悪化(副作用ではない) |
| 3列 + `City_Type` | 92 → 69 | -0.000425 | -13.70 | 悪化 |

**「効かない列」の派生列が、importance の 16.6% を占めていた。**

| 列 | importance の順位 | シェア |
|---|---|---|
| `te_Age` | **5 位** | 6.42% |
| `Age` | 12 位 | 3.13% |
| `cnt_Age` | 13 位 | 2.84% |

年齢の購入率の幅は全体で見ると 4.1 pt しかないが、**値ごとの細かい差**を TE が拾っていた。
**教訓: 目的変数との全体の関係が弱いことと、モデルにとって不要なことは別物。**

#### (4) 補助金との組み合わせ(仮説5)

**仮説**: 補助金がない人は、環境意識・収入・航続距離の不安のどれを動かしても購入率が床(0% 付近)に張り付く。
補助金との組み合わせを明示すれば効くのではないか。

| 作り方 | 関数・パターン | 列数 | 変化 | z | 判定 |
|---|---|---|---|---|---|
| 組み合わせをキーにした TE | `te2subsidy` | 92 → 101 | -0.000016 | -0.94 | 誤差 |
| 補助金(0/1)× 各列の積 | `add_subsidy_products` | 92 → 95 | +0.000009 | +0.65 | 誤差 |

**「床に張り付く」のは確率の見た目で、log-odds では足し算だった。**
補助金の効果は log-odds で見るとどの水準でも +4.06〜4.49 でほぼ一定。
木は `Subsidy_Available` で1回分割すれば同じ構造を表現できる。
組み合わせ列は importance 13.6% とよく使われたが、単体の補助金列の出番が減っただけで AUC は動かなかった。

#### (5) NN(RealMLP)だけが組み合わせを取りこぼしていないか

**仮説**: 木は条件分岐が得意だが、NN は苦手かもしれない。GBDT には不要でも、RealMLP には効くのではないか。

**検証**: EDA で見つけた構造ごとに行を分け、各モデルの予測を観測の購入率に合わせて補正したとき、
AUC が上がるかを測った(上がれば、その構造を取りこぼしている)。単位は 0.00001、採用基準は +8。

| 構造 | LightGBM | XGBoost | CatBoost | RealMLP |
|---|---|---|---|---|
| 自宅充電 × 自宅スタンド数 | -1.1 | -1.1 | -0.9 | -0.9 |
| 都市区分 × 自宅スタンド数 | -1.1 | -1.1 | -1.1 | -1.1 |
| 補助金 × 環境意識 | -1.5 | -1.4 | -1.0 | -4.1 |
| 補助金 × 収入 | -1.2 | -1.3 | -1.1 | -4.9 |
| 対照(意味のないランダム 30 群) | -0.1 | -0.1 | -0.1 | -0.1 |

**すべての構造・すべてのモデルで、補正すると AUC はむしろ下がった。** RealMLP も取りこぼしていない。
診断の感度は、予測をわざと壊して回復できるかで確認済み(補助金の効果を半分消すと、98% を回復できた)。

### 8-2 四則演算 — パターン別

EDA とは別に、Feature Engineering の定番として試した。
**演算ごとには分けて測っておらず、下の組み合わせで束ねて投入した結果**である。

**試したパターン**

| パターン | 作った列(例) | 狙い |
|---|---|---|
| 和(sum) | 自宅 + 職場のスタンド数 / 数値列の全ペアの和 | 充電環境の総量のような「合計」に意味があるか |
| 差(diff) | 自宅 − 職場のスタンド数 / 数値列の全ペアの差 | 2つの量のバランスに意味があるか |
| 比(ratio) | 年収 ÷ 通勤距離、年収 ÷ 年齢、年収 ÷ 保有台数、通勤距離 ÷ スタンド数、自宅 ÷ 職場スタンド数 | 「1台あたり」「1 km あたり」のような単位あたりの量 |
| 積(product) | 年収 × 環境意識、年齢 × 環境意識、通勤距離 × 年齢、環境意識 × スタンド数 | 2つの効果が掛け算で効くか |
| 平均(avg) | (自宅 + 職場) ÷ 2 / 7ペアの平均 | 和と同じ(木にとっては和を 2 で割っても区別がつかない) |
| 行方向の集約 | 数値列を標準化した平均・標準偏差 / スタンド数の最小・最大・合計 | 行全体の「高さ」「ばらつき」 |

**測定結果**

| 関数 | ファイル | 含むパターン | 列数 | 変化 | 判定 |
|---|---|---|---|---|---|
| `add_arithmetic_meaningful` | lgbm | 和・差・比・積(意味で選んだ12列) | 12 | -0.00014 → 本番構成で再測定 -0.000048(z=-2.58) | 誤差 |
| `add_arithmetic_all_pairs` | lgbm | 和・差・比 × 数値7列の全21ペア | 63 | +0.00030(3-fold)→ 再測定 **-0.000089(z=-4.99)** | **悪化** |
| `add_arithmetic` / `all_numeric_pairs` | xgb | 和・差・比・平均 × 意味で選んだ7ペア | 28 | +0.00004 → 再測定 -0.000049(z=-3.22) | 誤差 |
| `add_arithmetic` | catboost | 和・差・比・積・平均(14列) | 14 | **-0.00099** | **悪化**(最大) |
| `arithmetic_meaningful` | all | 上の集約版 | — | — | 同上 |
| `add_group_means` | lgbm | 行方向の平均・標準偏差 | 3 | 効果なし | 不採用 |
| `add_row_aggregates` | xgb | スタンド数の最小・最大・合計 | 3 | 効果なし | 不採用 |

**3モデルとも、どのパターンでも効かなかった。** EDA のとおり各列の効果は log-odds 上の足し算で、
「比」や「積」の関係は生成過程に入っていないと考えられる。
CatBoost が最も悪化したのは、対称木(同じ深さでは全ノードが同じ条件で分割する)が不要な列の影響を受けやすいため(推測)。

### 8-3 交互作用 TE・行フィンガープリント(EDA 以前に試したもの)

| 関数 | ファイル | 内容 | 変化 | 理由 |
|---|---|---|---|---|
| `pair_keys` | lgbm | カテゴリ6列の全15ペアで TE | -0.00007 | 単列の TE と木の分割で取り切れている |
| `triple_keys` | lgbm | 3列の組み合わせ5組で TE(8-1 の組を含む) | -0.00004 | 2列 → 3列 → 6列 → 13列すべて無効 |
| `make_interaction_keys` / `cat_pairs` | xgb | 2列の組み合わせキー | 無効 | 同上 |
| `add_interactions` | catboost | カテゴリの組み合わせ(10組) | -0.00008 | CatBoost は内部でも組み合わせの統計を作るため冗長 |
| `interaction_keys` / `cat_pairs` | all | 上の集約版 | — | 同上 |
| `all_columns_key` | lgbm | 全13列の組み合わせ(行フィンガープリント) | 実行前に打ち切り | train の全行がユニークで原理的に機能しない(前コンペ S6E8 では最大の改善要因だった) |
| `fingerprint_key` | lgbm | 列の部分集合での組み合わせ | -0.00002 | 多列の組み合わせ自体に追加情報がない |

### 8-4 エンコード方式・Count Encoding

| 関数 | ファイル | 内容 | 変化 | 理由 |
|---|---|---|---|---|
| `as_native_category` | xgb | カテゴリ列を category 型で渡す | 同点 | ordinal と差がなく、LightGBM との違いを出すため ordinal を採用(5-2) |
| `as_onehot` | xgb | One-Hot Encoding | +0.00005 → 再測定 -0.000002 | 列が増えるだけ。カテゴリは 2〜4 種類しかない |
| `add_count_encoding` | catboost | Count Encoding | -0.00017 | CatBoost 内部の統計と重複(4-2) |

### 8-5 Target Encoding の旧版 — より良い版に置き換え

| 関数 | ファイル | 内容 | 変化 | 理由 |
|---|---|---|---|---|
| `fit_target_encoding` / `apply_target_encoding` / `fit_apply_te_cv` | xgb | 入れ子にしない TE | -0.00108(入れ子版との差) | 学習行に自分の正解が混ざり、当たりすぎる(4-1) |
| `fit_apply_te_cv_nested` | xgb | 入れ子 TE(平滑化1種類) | — | 平滑化3種の同時投入に置き換え |
| `fit_target_encoding_multi` / `apply_target_encoding_multi` | xgb | 入れ子にしない平滑化3種の TE | — | 同上 |

### 8-6 RealMLP への GBDT 系の情報の追加 — 単体は伸びてもアンサンブルで消える

| 関数 | ファイル | 内容 | 変化 | 理由 |
|---|---|---|---|---|
| `build_te_key_frame` / `target_encode_highcard` | realmlp | 高カーディナリティ2列の厳密値 TE | 単体 +0.000156 / アンサンブル ±0 | GBDT と予測が似て、多様性が減る(5-4) |
| `add_income_neighborhood` | all | 年収 ±r ドルの範囲の購入率・傾き・曲率 | GBDT ±0.00002 / RealMLP 単体 +0.000077(z=+3.45)/ アンサンブル -0.0000019 | 同上。Smooth Keys と厳密値 TE で取り込み済み |